# Current debugging file for validating difference between ML and QA
- Validating outputted models work as intended
- Troubleshooting

In [ ]:
import onnx
import numpy as np
from onnx import helper, numpy_helper, TensorProto
import onnxruntime as ort
import Utils
import matplotlib.pyplot as plt
import importlib
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
plt.rcParams["axes.grid"] = True
# plt.yscale("log")

In [ ]:
# INPUTFILE = [f"../ML-debug/AnalysisResults-ML/FwdMatchMLCandidates-{i}.root" for i in range(1, 21)]
# print(INPUTFILE)
INPUTFILE = "FwdMatchMLCandidates_Troubleshoot0306.root"
MODELFILE = "model_all_features_positive.onnx"

In [ ]:
FEATURES = ['XMCH',
 'YMCH',
 'PhiMCH',
 'TanlMCH',
 'InvQPtMCH',
 'Chi2MCH',
 'PDCA',
 'Rabs',
 'CXXMCH',
 'CYYMCH',
 'CPhiPhiMCH',
 'CTglTglMCH',
 'C1Pt1PtMCH',
 'CXYMCH',
 'CPhiYMCH',
 'CPhiXMCH',
 'CTglXMCH',
 'CTglYMCH',
 'CTglPhiMCH',
 'C1PtXMCH',
 'C1PtYMCH',
 'C1PtPhiMCH',
 'C1PtTglMCH',
 'XMFT',
 'YMFT',
 'PhiMFT',
 'TanlMFT',
 'InvQPtMFT',
 'Chi2MFT',
 'TrackTypeMFT',
 'CXXMFT',
 'CYYMFT',
 'CPhiPhiMFT',
 'CTglTglMFT',
 'C1Pt1PtMFT',
 'CXYMFT',
 'CPhiYMFT',
 'CPhiXMFT',
 'CTglXMFT',
 'CTglYMFT',
 'CTglPhiMFT',
 'C1PtXMFT',
 'C1PtYMFT',
 'C1PtPhiMFT',
 'C1PtTglMFT',
 'DCAX',
 'DCAY',
 'IsAmbig',
 'MFTMult',
 'etaMCH',
 'etaMFT',
 'DeltaEta',
 'DeltaX',
 'DeltaY',
 'DeltaPhi',
 'DeltaTanl',
 'DeltaR',
 'SameSign',
 'PtMCH',
 'PtMFT',
 'DeltaPt',
 'RelPtDiff',
 'PullPt',
 'PullX',
 'PullY',
 'PullR',
 'PullPhi',
 'PullTanl',
 'DeltaDirection']

In [ ]:
df = Utils.get_dataframe(INPUTFILE, folder_name="DF_*")
df = Utils.process_dataframe(df, makedummies=False)


In [ ]:
df.describe()

In [ ]:
sess = ort.InferenceSession(MODELFILE)
input_name = sess.get_inputs()[0].name

In [ ]:
df['score'] = sess.run(
    None,
    {input_name: df[FEATURES].to_numpy(dtype=np.float32)}
)[0].ravel()# used for the binary classifier, for the classifier we want the fuller array

In [ ]:
for output in sess.get_outputs():
    print(output.name, output.shape, output.type)

In [ ]:
df.describe()

In [ ]:
import matplotlib.pyplot as plt
importlib.reload(plt)

old_subplots = plt.subplots

def subplots_custom(*args, **kwargs):
    fig, ax = old_subplots(*args, **kwargs)

    def apply(a):
        a.set_yscale("log")
        a.set_ylim(bottom=0.5)
        a.set_ylim(top=1000000)
        a.grid(True, alpha=0.3, linestyle="--")

    if isinstance(ax, (list, tuple)):
        for a in ax:
            apply(a)
    else:
        apply(ax)

    return fig, ax

plt.subplots = subplots_custom

In [ ]:
# #Hack append some high score wrong matches to force the bins to work as in the qa-task
# df_new = df.copy()
# for i in range(0,4): 
#     # Copy a single row as a DataFrame
#     df_testing = df[df["MatchLabel"] == i].iloc[[1]].copy()

#     # Modify the score column
#     df_testing["score"] = 1.0

#     # Append back
#     df_new = pd.concat([df_new, df_testing], ignore_index=True)

In [ ]:
match_groups = Utils.build_match_groups(df)

Utils.draw_feature("score", match_groups=match_groups, density=False)

In [ ]:
df_leader = df.loc[df.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=["score"], match_groups=match_groups_leader, density=False, per=0.0)

In [ ]:
print(df[FEATURES+["score"]].head(100).to_string())  

In [ ]:
df[FEATURES+["score"]].head(100).to_csv("onnx_validation_output.csv", index=True)

In [ ]:
df[df['XMCH'] ==3.9083308]